In [ ]:
# =============================================================================================
# Questo notebook calcola le ampiezze di elicità leptoniche utilizzando
# che verranno usate nel formalismo di elicità per la costruzione dell'ampiezza di decadimento
# =============================================================================================

# --- Import delle librerie necessarie ---
import sympy as sp
from sympy.assumptions import Q

# Configurazione per la stampa formattata delle espressioni
sp.init_printing(use_unicode=True, wrap_line=True, num_columns=120)

# --- Variabili cinematiche ---
q0, qz, q2 = sp.symbols('q0 qz q2', real=True)  # Componenti quadri-impulso (time-like, z e q^2)
m = sp.symbols('m', positive=True, real=True)  # Massa del leptone
E = sp.symbols('E', real=True, positive=True)  # Energia del leptone

# --- Definizione dei parametri beta± ---
beta_p = sp.sqrt(E + m)  # beta_plus = √(E + m)
beta_m = sp.sqrt(E - m)  # beta_minus = √(E - m)

# --- Imposizione delle assunzioni fisiche ---
# Forza le assunzioni: E > m > 0, che implica E+m > 0 e E-m > 0
# Questo è necessario per garantire che le radici quadrate siano reali e positive
assumptions = Q.positive(E) & Q.positive(m) & Q.gt(E, m)

# Semplifica beta± con le assunzioni
beta_p = sp.refine(beta_p, assumptions)
beta_m = sp.refine(beta_m, assumptions)


In [ ]:
# ============================================================================
# DEFINIZIONE DEI VETTORI DI POLARIZZAZIONE
# ============================================================================

# ω_+(+1):
omega_p = sp.Matrix([0, -1, -sp.I, 0])/sp.sqrt(2)

# ω_-(-1):
omega_m = sp.Matrix([0, +1, -sp.I, 0])/sp.sqrt(2)

# ω_0(0): 
omega_0 = sp.Matrix([0, 0, 0, 1])

# ω_t: 
omega_t = sp.Matrix([1, 0, 0, 0])


In [ ]:
# ============================================================================
# DEFINIZIONE DELLE MATRICI GAMMA E COSTRUZIONE DEGLI SPINORI
# ============================================================================
# Implementazione delle matrici di Dirac nella rappresentazione di Dirac
# e costruzione degli spinori u(λ) e v(λ) per particelle e antiparticelle

# --- Matrici di Pauli ---
I2 = sp.eye(2)  # Matrice identità 2×2
sigma1 = sp.Matrix([[0,1],[1,0]])      # σ_x
sigma2 = sp.Matrix([[0,-sp.I],[sp.I,0]])  # σ_y
sigma3 = sp.Matrix([[1,0],[0,-1]])     # σ_z

# --- Matrici gamma nella rappresentazione di Dirac ---
# γ^0 = [[I, 0], [0, -I]]
gamma0 = sp.Matrix(sp.BlockMatrix([[sp.eye(2), sp.zeros(2)],[sp.zeros(2), -sp.eye(2)]]))

# γ^i = [[0, σ_i], [-σ_i, 0]] per i = 1,2,3
gamma1 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sigma1],[ -sigma1, sp.zeros(2)]]))
gamma2 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sigma2],[ -sigma2, sp.zeros(2)]]))
gamma3 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sigma3],[ -sigma3, sp.zeros(2)]]))

# γ^5 = [[0, I], [I, 0]]
gamma5 = sp.Matrix(sp.BlockMatrix([[sp.zeros(2), sp.eye(2)],[sp.eye(2), sp.zeros(2)]]))

# Vettore di matrici gamma per iterazioni
gamma = [gamma0, gamma1, gamma2, gamma3]

# --- Definizione degli spinori u(λ) e v(λ) ---
# Componenti ordinate come vettori colonna 4×1

# Spinori per particelle
u_u_p = sp.Matrix([beta_p, 0, beta_m, 0])    # u(+1/2): elicità positiva
u_u_m = sp.Matrix([0, beta_p, 0, -beta_m])   # u(-1/2): elicità negativa

# Spinori per antiparticelle
v_v_p = sp.Matrix([beta_m, 0, -beta_p, 0])   # v(+1/2): elicità positiva
v_v_m = sp.Matrix([0, beta_m, 0, beta_p])    # v(-1/2): elicità negativa

# --- Funzioni helper per accedere agli spinori ---
def u_spinor(hel):
    """
    Restituisce lo spinore u per l'elicità hel (+1/2 o -1/2).
    
    Args:
        hel: Elicità del leptone (+1/2 o -1/2)
        
    Returns:
        Vettore colonna 4x1 dello spinore u
    """
    if hel == 1/2 or hel == sp.Rational(1,2):
        return u_u_p
    elif hel == -1/2 or hel == sp.Rational(-1,2):
        return u_u_m
    else:
        raise ValueError("hel must be +1/2 or -1/2")

def v_spinor(hel):
    """
    Restituisce lo spinore v per l'elicità hel (+1/2 o -1/2).
    
    Args:
        hel: Elicità dell'antileptone (+1/2 o -1/2)
        
    Returns:
        Vettore colonna 4x1 dello spinore v
    """
    if hel == 1/2 or hel == sp.Rational(1,2):
        return v_v_p
    elif hel == -1/2 or hel == sp.Rational(-1,2):
        return v_v_m
    else:
        raise ValueError("hel must be +1/2 or -1/2")

# --- Definizione di ubar e vbar ---
# ubar = u^† γ^0 (spinore aggiunto di Dirac)
def ubar(u):
    """Calcola lo spinore aggiunto ū """
    return (u.conjugate().T * gamma0)

def vbar(v):
    """Calcola lo spinore aggiunto v̄ """
    return (v.conjugate().T * gamma0)

# --- Funzioni per calcolare le ampiezze leptoniche ---

def L_vector(u, v, omega):
    """
    Calcola l'ampiezza leptonica vettoriale L_V
    
    Args:
        u: Spinore u del leptone
        v: Spinore v dell'antileptone
        omega: Vettore di polarizzazione
        
    Returns:
        Ampiezza leptonica vettoriale (scalare complesso)
    """
    res = sp.Rational(0)
    g = [1, -1, -1, -1]  # Metrica di Minkowski diag(+,-,-,-)
    for mu in range(4):
        res += (ubar(u) * gamma[mu] * v)[0] * g[mu] * omega[mu]
    return sp.simplify(res)

def L_axial(u, v, omega):
    """
    Calcola l'ampiezza leptonica assiale L_A
    
    Args:
        u: Spinore u del leptone
        v: Spinore v dell'antileptone
        omega: Vettore di polarizzazione
        
    Returns:
        Ampiezza leptonica assiale (scalare complesso)
    """
    res = sp.Rational(0)
    g = [1, -1, -1, -1]  # Metrica di Minkowski
    for mu in range(4):
        res += (ubar(u) * gamma[mu] * gamma5 * v)[0] * g[mu] * omega[mu]
    return sp.simplify(res)

def build_L_matrices(omega=None):
    """
    Costruisce le matrici 2x2 delle ampiezze leptoniche L_V e L_A.
    
    Gli indici delle matrici corrispondono alle elicità:
    - Indice 0: λ = -1/2
    - Indice 1: λ = +1/2
    
    Args:
        omega: Vettore di polarizzazione da usare. Se None, viene scelto
               automaticamente in base a λ1 - λ2
               
    Returns:
        tuple: (LV_mat, LA_mat) matrici 2x2 delle ampiezze vettoriali e assiali
    """
    hel_vals = [sp.Rational(-1,2), sp.Rational(1,2)]
    LV_mat = sp.Matrix([[0,0],[0,0]])
    LA_mat = sp.Matrix([[0,0],[0,0]])

    def pick_omega_from_diff(lam_diff):
        """
        Seleziona il vettore di polarizzazione appropriato in base a Δλ = λ1 - λ2
        
        Args:
            lam_diff: Differenza di elicità λ1 - λ2
            
        Returns:
            Vettore di polarizzazione corrispondente
        """
        if lam_diff == sp.Integer(1):
            return omega_p  # Δλ = +1 → polarizzazione +
        elif lam_diff == sp.Integer(-1):
            return omega_m  # Δλ = -1 → polarizzazione -
        elif lam_diff == sp.Integer(0):
            return omega_0  # Δλ = 0 → polarizzazione longitudinale
        else:
            return omega_t  # Fallback: polarizzazione timelike

    # Calcola tutti gli elementi delle matrici
    for i, lam1 in enumerate(hel_vals):
        for j, lam2 in enumerate(hel_vals):
            lam_diff = lam1 - lam2  # Differenza di elicità
            # Sceglie la polarizzazione: usa omega fornito o seleziona automaticamente
            chosen_omega = omega if omega is not None else pick_omega_from_diff(lam_diff)
            u = u_spinor(lam1)
            v = v_spinor(lam2)
            LV_mat[i, j] = sp.simplify(L_vector(u, v, chosen_omega))
            LA_mat[i, j] = sp.simplify(L_axial(u, v, chosen_omega))

    return LV_mat, LA_mat

# --- Costruzione delle matrici di ampiezze leptoniche ---
# Selezione automatica della polarizzazione in base a λ1 - λ2
LV, LA = build_L_matrices() 

def force_conjugate_simplification(expr):
    """
    Forza la semplificazione conjugate(√x) = √x per espressioni positive.
    
    Questo è necessario perché SymPy non sempre riconosce automaticamente che
    il coniugato di una radice quadrata di una quantità positiva è uguale a se stesso.
    
    Args:
        expr: Espressione SymPy da semplificare
        
    Returns:
        Espressione semplificata
    """
    # Sostituisce conjugate(√(E+m)) con √(E+m)
    expr = expr.replace(lambda e: e.func == sp.conjugate and 
                                  len(e.args) == 1 and
                                  e.args[0].func == sp.Pow and 
                                  e.args[0].exp == sp.Rational(1,2),
                       lambda e: e.args[0])
    return sp.simplify(expr)

# Applica la semplificazione elemento per elemento
LV = LV.applyfunc(force_conjugate_simplification)
LA = LA.applyfunc(force_conjugate_simplification)

# Applica refine con le assunzioni fisiche (E > m > 0)
LV = sp.refine(LV, assumptions)
LA = sp.refine(LA, assumptions)

# --- Sostituzioni cinematiche ---
q = sp.symbols('q', positive=True, real=True)  # Impulso del leptone

# Relazioni cinematiche
kinematic_subs = {
    E**2: q**2/4,
    E: sp.sqrt(q**2)/2
}

# Applica le sostituzioni cinematiche
LV = LV.subs(kinematic_subs)
LA = LA.subs(kinematic_subs)


In [ ]:
# ============================================================================
# AMPIEZZA LEPTONICA TIMELIKE
# ============================================================================

# Valori di elicità
hel_vals = [sp.Rational(-1,2), sp.Rational(1,2)]

# Inizializzazione della matrice 2×2
LT = sp.Matrix([[0,0],[0,0]])

# --- Calcolo degli elementi di matrice ---
for i, lam1 in enumerate(hel_vals):
    for j, lam2 in enumerate(hel_vals):
        u = u_spinor(lam1)   # Spinore del leptone con elicità λ1
        v = v_spinor(lam2)   # Spinore dell'antileptone con elicità λ2
        LT[i, j] = 2*m * (ubar(u) * gamma5 * v)[0]

# --- Applicazione delle semplificazioni ---
LT = LT.applyfunc(force_conjugate_simplification)  # Semplifica i coniugati
LT = sp.refine(LT, assumptions)  # Applica le assunzioni fisiche
LT = LT.subs(kinematic_subs)  # Applica le sostituzioni cinematiche
LT = sp.simplify(LT)  # Semplificazione finale


In [ ]:
# ============================================================================
# COSTRUZIONE DELLE AMPIEZZE LEFT-HANDED E RIGHT-HANDED
# ============================================================================
# Costruisce le combinazioni V±A che corrispondono alle correnti chirali:
# - L_Left  = L_V - L_A: corrente left-handed (chiralità negativa)
# - L_Right = L_V + L_A: corrente right-handed (chiralità positiva)

LLeft = LV - LA
LRight = LV + LA

# --- Visualizzazione delle ampiezze in formato LaTeX ---
from IPython.display import display, Math

# Visualizza le matrici delle ampiezze leptoniche
display(Math(r'\mathbf{L}_{\text{Vector}} = ' + sp.latex(LV)))
display(Math(r'\mathbf{L}_{\text{Axial}} = ' + sp.latex(LA)))

display(Math(r'\mathbf{L}_{\text{Left}} = ' + sp.latex(LLeft)))
display(Math(r'\mathbf{L}_{\text{Right}} = ' + sp.latex(LRight)))

display(Math(r'\mathbf{L}_{\text{Timelike}} = ' + sp.latex(LT)))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>